In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# PhishGuard AI — STAGE 0: Setup & Dual-Pipeline Cleaning
# Saves: train.csv, val.csv, test.csv, experiment_config.csv → NLP FINAL/dataset/
# ═══════════════════════════════════════════════════════════════════════════════

# ═══ CELL 1 — Reproducibility Block (required on every stage) ═══
import os, random, numpy as np, torch

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True, warn_only=True)

print("✓ Reproducibility block applied (SEED=42)")


# ═══ CELL 2 — Drive mount + folder tree ═══
from google.colab import drive
drive.mount("/content/drive")

BASE_DIR     = "/content/drive/MyDrive/NLP FINAL"
RAW_DIR      = os.path.join(BASE_DIR, "raw dataset")
DATASET_DIR  = os.path.join(BASE_DIR, "dataset")
STAGE0_DIR   = os.path.join(BASE_DIR, "stage0_setup")

for folder in [RAW_DIR, DATASET_DIR, STAGE0_DIR,
               os.path.join(STAGE0_DIR, "models"),
               os.path.join(STAGE0_DIR, "logs"),
               os.path.join(STAGE0_DIR, "results"),
               os.path.join(STAGE0_DIR, "diagrams")]:
    os.makedirs(folder, exist_ok=True)

# Try your actual Drive path first, then guideline fallback
RAW_CANDIDATES = [
    os.path.join(RAW_DIR, "Phishing_Email.csv"),
    os.path.join(DATASET_DIR, "Phishing_Email.csv"),
    os.path.join(BASE_DIR, "dataset", "Phishing_Email.csv"),
]
RAW_PATH = next((p for p in RAW_CANDIDATES if os.path.exists(p)), None)
if RAW_PATH is None:
    raise FileNotFoundError(
        "Phishing_Email.csv not found. Expected at:\n  "
        + "\n  ".join(RAW_CANDIDATES)
    )

print(f"✓ BASE_DIR  : {BASE_DIR}")
print(f"✓ RAW_PATH  : {RAW_PATH}")
print(f"✓ OUTPUT_DIR: {DATASET_DIR}")


# ═══ CELL 3 — Install dependencies + NLTK resources ═══
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "beautifulsoup4", "lxml", "scikit-learn"])

import nltk
for resource in ["punkt", "punkt_tab", "stopwords", "wordnet",
                 "omw-1.4", "averaged_perceptron_tagger",
                 "averaged_perceptron_tagger_eng"]:
    try:
        nltk.download(resource, quiet=True)
    except Exception:
        pass  # some resources renamed across NLTK versions; core ones still load

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

STOP_WORDS  = set(stopwords.words("english"))
LEMMATIZER  = WordNetLemmatizer()
print(f"✓ NLTK ready — {len(STOP_WORDS)} English stopwords loaded")


# ═══ CELL 4 — Cleaning functions (dual pipeline) ═══
import re
import pandas as pd
from bs4 import BeautifulSoup

URL_PATTERN = re.compile(r"https?://\S+|www\.\S+", re.IGNORECASE)
MAX_CHARS   = 100_000   # cap extreme outliers (~17M char row) — logged in config

LABEL_MAP = {
    "Safe Email":     0,   # Legitimate
    "Phishing Email": 1,   # Phishing
}


def strip_html(text: str) -> str:
    if not isinstance(text, str):
        return ""
    return BeautifulSoup(text, "lxml").get_text(separator=" ")


def replace_urls(text: str) -> str:
    return URL_PATTERN.sub("<URL>", text)


def normalize_whitespace(text: str) -> str:
    return re.sub(r"\s+", " ", text).strip()


def truncate(text: str, max_chars: int = MAX_CHARS) -> str:
    return text[:max_chars] if len(text) > max_chars else text


def clean_classical(raw_text: str) -> str:
    """
    Heavy cleaning for TF-IDF / classical ML (RQ4):
    lowercase → strip HTML → URL token → remove punctuation
    → stopword removal → lemmatization
    """
    text = truncate(str(raw_text))
    text = strip_html(text)
    text = replace_urls(text)
    text = text.lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    text = normalize_whitespace(text)
    words = [
        LEMMATIZER.lemmatize(w)
        for w in text.split()
        if w not in STOP_WORDS and len(w) > 1
    ]
    return " ".join(words)


def clean_transformer(raw_text: str) -> str:
    """
    Light cleaning for BiLSTM / TextCNN / DistilBERT (RQ5, RQ6):
    strip HTML → URL token → preserve case, punctuation, sentence structure
    """
    text = truncate(str(raw_text))
    text = strip_html(text)
    text = replace_urls(text)
    return normalize_whitespace(text)


print("✓ Dual cleaning functions defined")


# ═══ CELL 5 — Load raw data ═══
raw_df = pd.read_csv(RAW_PATH)
print(f"Raw shape: {raw_df.shape}")
print(f"Raw columns: {list(raw_df.columns)}")

# Drop Kaggle index column if present
if "Unnamed: 0" in raw_df.columns:
    raw_df = raw_df.drop(columns=["Unnamed: 0"])

assert "Email Text" in raw_df.columns, "Missing column: Email Text"
assert "Email Type"  in raw_df.columns, "Missing column: Email Type"

raw_df["Email Text"] = raw_df["Email Text"].astype(str)
raw_df.loc[raw_df["Email Text"].isin(["nan", "None"]), "Email Text"] = np.nan

print("\nRaw label distribution:")
print(raw_df["Email Type"].value_counts())


# ═══ CELL 6 — Filter + label encode ═══
df = raw_df.copy()

# Remove null / empty / whitespace-only emails (matches plan: 18,650 → 18,634)
df = df.dropna(subset=["Email Text"])
df = df[df["Email Text"].str.strip().astype(bool)].copy()

# Encode labels
unknown = set(df["Email Type"].unique()) - set(LABEL_MAP.keys())
if unknown:
    raise ValueError(f"Unexpected labels found: {unknown}")

df["label"] = df["Email Type"].map(LABEL_MAP).astype(int)
df = df.reset_index(drop=True)

print(f"After null/empty removal: {len(df):,} rows")
print(df["label"].value_counts().rename({0: "Safe (0)", 1: "Phishing (1)"}))


# ═══ CELL 7 — Apply dual preprocessing pipelines ═══
print("\nApplying dual preprocessing (this takes ~5–15 min on 18K emails)...")

df["text_cleaned_classical"]   = df["Email Text"].apply(clean_classical)
df["text_cleaned_transformer"] = df["Email Text"].apply(clean_transformer)

# Drop rows where transformer pipeline produced empty text
before = len(df)
df = df[df["text_cleaned_transformer"].str.strip().astype(bool)].copy()
dropped_empty = before - len(df)
if dropped_empty:
    print(f"⚠ Dropped {dropped_empty} rows with empty transformer text after cleaning")

df = df.reset_index(drop=True)
print(f"Final cleaned dataset: {len(df):,} rows")


# ═══ CELL 8 — Stratified 70 / 15 / 15 split (ONCE — never repeat) ═══
from sklearn.model_selection import train_test_split

TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
TEST_RATIO  = 0.15
assert abs(TRAIN_RATIO + VAL_RATIO + TEST_RATIO - 1.0) < 1e-9

# 70% train | 30% temp  →  then temp → 50/50 → 15% val + 15% test
train_df, temp_df = train_test_split(
    df,
    test_size=(VAL_RATIO + TEST_RATIO),
    random_state=SEED,
    stratify=df["label"],
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=SEED,
    stratify=temp_df["label"],
)

OUTPUT_COLS = ["text_cleaned_classical", "text_cleaned_transformer", "label"]

train_out = train_df[OUTPUT_COLS].reset_index(drop=True)
val_out   = val_df[OUTPUT_COLS].reset_index(drop=True)
test_out  = test_df[OUTPUT_COLS].reset_index(drop=True)

print("\nSplit sizes:")
print(f"  Train : {len(train_out):,}  ({len(train_out)/len(df)*100:.1f}%)")
print(f"  Val   : {len(val_out):,}  ({len(val_out)/len(df)*100:.1f}%)")
print(f"  Test  : {len(test_out):,}  ({len(test_out)/len(df)*100:.1f}%)")

for name, split in [("Train", train_out), ("Val", val_out), ("Test", test_out)]:
    safe_ct    = (split["label"] == 0).sum()
    phish_ct   = (split["label"] == 1).sum()
    print(f"  {name} labels → Safe: {safe_ct:,} | Phishing: {phish_ct:,}")


# ═══ CELL 9 — Save CSVs to Drive ═══
train_path = os.path.join(DATASET_DIR, "train.csv")
val_path   = os.path.join(DATASET_DIR, "val.csv")
test_path  = os.path.join(DATASET_DIR, "test.csv")

train_out.to_csv(train_path, index=False)
val_out.to_csv(val_path,   index=False)
test_out.to_csv(test_path,  index=False)

print("\n✓ Saved:")
print(f"  {train_path}")
print(f"  {val_path}")
print(f"  {test_path}")


# ═══ CELL 10 — Save experiment_config.csv ═══
import sklearn
import platform
from datetime import datetime

def split_stats(split_df, split_name):
    return {
        f"{split_name}_total":        len(split_df),
        f"{split_name}_safe_count":   int((split_df["label"] == 0).sum()),
        f"{split_name}_phishing_count": int((split_df["label"] == 1).sum()),
        f"{split_name}_safe_pct":     round((split_df["label"] == 0).mean() * 100, 2),
        f"{split_name}_phishing_pct": round((split_df["label"] == 1).mean() * 100, 2),
    }

config = {
    "project":              "PhishGuard AI",
    "stage":                "Stage 0 — Setup & Dual-Pipeline Cleaning",
    "timestamp_utc":        datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S"),
    "seed":                 SEED,
    "train_ratio":          TRAIN_RATIO,
    "val_ratio":            VAL_RATIO,
    "test_ratio":           TEST_RATIO,
    "stratified":           True,
    "raw_file":             RAW_PATH,
    "raw_rows":             len(raw_df),
    "rows_after_null_drop": len(raw_df.dropna(subset=["Email Text"])),
    "rows_final":           len(df),
    "rows_dropped_empty_transformer": dropped_empty,
    "max_char_truncation":  MAX_CHARS,
    "label_0":              "Safe Email (Legitimate)",
    "label_1":              "Phishing Email (Phishing)",
    "classical_pipeline":   "lowercase | strip_html | url_to_<URL> | remove_punctuation | stopwords | lemmatization",
    "transformer_pipeline": "strip_html | url_to_<URL> | preserve_case_punctuation_structure",
    "output_columns":       ", ".join(OUTPUT_COLS),
    "python_version":       platform.python_version(),
    "pandas_version":       pd.__version__,
    "numpy_version":        np.__version__,
    "sklearn_version":      sklearn.__version__,
    "torch_version":        torch.__version__,
    "nltk_version":         nltk.__version__,
}

for split_name, split_df in [("train", train_out), ("val", val_out), ("test", test_out)]:
    config.update(split_stats(split_df, split_name))

config_df = pd.DataFrame(list(config.items()), columns=["parameter", "value"])

config_dataset = os.path.join(DATASET_DIR, "experiment_config.csv")
config_stage0    = os.path.join(STAGE0_DIR, "results", "experiment_config.csv")

config_df.to_csv(config_dataset, index=False)
config_df.to_csv(config_stage0,   index=False)

print("\n✓ experiment_config.csv saved to:")
print(f"  {config_dataset}")
print(f"  {config_stage0}")


# ═══ CELL 11 — Final verification ═══
print("\n" + "=" * 70)
print("STAGE 0 COMPLETE ✓")
print("=" * 70)
print(f"Total emails processed : {len(df):,}")
print(f"  Safe (0)               : {(df['label']==0).sum():,}")
print(f"  Phishing (1)           : {(df['label']==1).sum():,}")
print(f"\nTrain / Val / Test       : {len(train_out):,} / {len(val_out):,} / {len(test_out):,}")
print("\nSample classical text (first train row, 120 chars):")
print(" ", train_out["text_cleaned_classical"].iloc[0][:120], "...")
print("\nSample transformer text (first train row, 120 chars):")
print(" ", train_out["text_cleaned_transformer"].iloc[0][:120], "...")
print("\nNext step → RQ1 (EDA) loads:", train_path)
print("=" * 70)

✓ Reproducibility block applied (SEED=42)
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ BASE_DIR  : /content/drive/MyDrive/NLP FINAL
✓ RAW_PATH  : /content/drive/MyDrive/NLP FINAL/raw dataset/Phishing_Email.csv
✓ OUTPUT_DIR: /content/drive/MyDrive/NLP FINAL/dataset
✓ NLTK ready — 198 English stopwords loaded
✓ Dual cleaning functions defined
Raw shape: (18650, 3)
Raw columns: ['Unnamed: 0', 'Email Text', 'Email Type']

Raw label distribution:
Email Type
Safe Email        11322
Phishing Email     7328
Name: count, dtype: int64
After null/empty removal: 18,631 rows
label
Safe (0)        11322
Phishing (1)     7309
Name: count, dtype: int64

Applying dual preprocessing (this takes ~5–15 min on 18K emails)...


/tmp/ipykernel_481/1858419271.py:98: MarkupResemblesLocatorWarning: The input passed in on this line looks more like a URL than HTML or XML.

If you meant to use Beautiful Soup to parse the web page found at a certain URL, then something has gone wrong. You should use an Python package like 'requests' to fetch the content behind the URL. Once you have the content as a string, you can feed that string into Beautiful Soup.

However, if you want to parse some data that happens to look like a URL, then nothing has gone wrong: you are using Beautiful Soup correctly, and this warning is spurious and can be filtered. To make this warning go away, run this code before calling the BeautifulSoup constructor:

    from bs4 import MarkupResemblesLocatorWarning
    import warnings

    warnings.filterwarnings("ignore", category=MarkupResemblesLocatorWarning)
    
  return BeautifulSoup(text, "lxml").get_text(separator=" ")
/tmp/ipykernel_481/1858419271.py:98: MarkupResemblesLocatorWarning: The inpu

Final cleaned dataset: 18,631 rows

Split sizes:
  Train : 13,041  (70.0%)
  Val   : 2,795  (15.0%)
  Test  : 2,795  (15.0%)
  Train labels → Safe: 7,925 | Phishing: 5,116
  Val labels → Safe: 1,698 | Phishing: 1,097
  Test labels → Safe: 1,699 | Phishing: 1,096

✓ Saved:
  /content/drive/MyDrive/NLP FINAL/dataset/train.csv
  /content/drive/MyDrive/NLP FINAL/dataset/val.csv
  /content/drive/MyDrive/NLP FINAL/dataset/test.csv

✓ experiment_config.csv saved to:
  /content/drive/MyDrive/NLP FINAL/dataset/experiment_config.csv
  /content/drive/MyDrive/NLP FINAL/stage0_setup/results/experiment_config.csv

STAGE 0 COMPLETE ✓
Total emails processed : 18,631
  Safe (0)               : 11,322
  Phishing (1)           : 7,309

Train / Val / Test       : 13,041 / 2,795 / 2,795

Sample classical text (first train row, 120 chars):
  interview dear m beck want thank interviewing analyst position last week enjoyed meeting learning work enthusiasm positi ...

Sample transformer text (first train row, 

/tmp/ipykernel_481/1858419271.py:274: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp_utc":        datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S"),
